In [1]:
import pandas as pd

In [2]:
inflation = pd.read_csv("../Data/Cleaned Data/Inflation_Cleaned.csv")
exchange = pd.read_csv("../Data/Cleaned Data/Exchange_Rate_Cleaned.csv")
remittances = pd.read_csv("../Data/Cleaned Data/remittances_clean.csv")
petrol = pd.read_csv("../Data/Cleaned Data/petrol_prices_clean.csv")

In [3]:
petrol.columns

Index(['Month', 'Petrol_Price'], dtype='object')

In [4]:
petrol.rename(columns={"Month": "Date"}, inplace=True)

In [5]:
inflation["Date"] = pd.to_datetime(inflation["Date"])
exchange["Date"] = pd.to_datetime(exchange["Date"])
remittances["Date"] = pd.to_datetime(remittances["Date"])
petrol["Date"] = pd.to_datetime(petrol["Date"])

In [6]:
inflation = inflation[inflation["Date"] >= "2010-01-01"]
exchange = exchange[exchange["Date"] >= "2010-01-01"]
remittances = remittances[remittances["Date"] >= "2010-01-01"]
petrol = petrol[petrol["Date"] >= "2010-01-01"]

In [7]:
inflation["Date"] = inflation["Date"].dt.to_period("M")
exchange["Date"] = exchange["Date"].dt.to_period("M")
remittances["Date"] = remittances["Date"].dt.to_period("M")
petrol["Date"] = petrol["Date"].dt.to_period("M")

In [8]:
master = inflation.merge(exchange, on="Date", how="left")
master = master.merge(remittances, on="Date", how="left")
master = master.merge(petrol, on="Date", how="left")

In [9]:
master.head()
master.info()
master.shape
master.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 198 entries, 0 to 197
Data columns (total 5 columns):
 #   Column                   Non-Null Count  Dtype    
---  ------                   --------------  -----    
 0   Date                     198 non-null    period[M]
 1   Inflation_YoY            198 non-null    float64  
 2   USD_PKR                  198 non-null    float64  
 3   Remittances_Million_USD  198 non-null    float64  
 4   Petrol_Price             198 non-null    float64  
dtypes: float64(4), period[M](1)
memory usage: 7.9 KB


Date                       0
Inflation_YoY              0
USD_PKR                    0
Remittances_Million_USD    0
Petrol_Price               0
dtype: int64

In [10]:
master["Date"] = master["Date"].dt.to_timestamp("M")

In [11]:
master = master.sort_values("Date")
master.reset_index(drop=True, inplace=True)

In [12]:
master.to_csv(
    "../Data/Cleaned Data/master_dataset.csv",
    index=False
)

In [13]:
master["Year"] = master["Date"].dt.year

In [14]:
master["Month_Number"] = master["Date"].dt.month

In [15]:
master["Month_Name"] = master["Date"].dt.month_name()

In [16]:
master["Quarter"] = master["Date"].dt.quarter

In [17]:
master["Fiscal_Year"] = master["Year"]

master.loc[
    master["Month_Number"] >= 7,
    "Fiscal_Year"
] += 1

In [18]:
master["Inflation_Change"] = master["Inflation_YoY"].diff()

In [19]:
master["USD_Change"] = master["USD_PKR"].diff()

In [20]:
master["Petrol_Change"] = master["Petrol_Price"].diff()

In [21]:
master["Remittance_Growth_%"] = (
    master["Remittances_Million_USD"].pct_change() * 100
)

In [22]:
master["USD_3M_MA"] = master["USD_PKR"].rolling(3).mean()

In [23]:
master["Petrol_3M_MA"] = master["Petrol_Price"].rolling(3).mean()

In [24]:
master["Remittances_3M_MA"] = (
    master["Remittances_Million_USD"]
    .rolling(3)
    .mean()
)

In [25]:
master["Yearly_USD_Avg"] = (
    master.groupby("Year")["USD_PKR"]
    .transform("mean")
)

In [26]:
master["Yearly_Petrol_Avg"] = (
    master.groupby("Year")["Petrol_Price"]
    .transform("mean")
)

In [27]:
master.to_csv(
    "../Data/Cleaned Data/master_dataset.csv",
    index=False
)